Section 1 : Impoting Libraries

In [1]:
import pandas as pd
import joblib

from scipy.sparse import csr_matrix, hstack, save_npz

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

Section 2 : Loading dataset

In [2]:
df = pd.read_csv(
    "../data/processed/feature_engineered_dataset.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (7419, 21)


,City,Listing Type,Field,Job Title,Company Name,Salary/Stipend,Job Description,Location,Posted Date,Experience Level,...,Real/Fake Job,combined_text,label,text_length,word_count,avg_word_length,keyword_risk_score,salary_risk_score,domain,domain_frequency
0,Mumbai,Full-time,Tech,Software Developer,Rao IIT Academy,"₹500,000 - ₹1,200,000",as a software developer at rao iit academy you...,"Mumbai, Maharashtra",2019-05-17T21:02:37Z,Entry Level (0-2 years),...,Real Job,software developer rao iit academy tech entry ...,0,550,89,5.191011,0,1,www.adzuna.in,6553
1,Mumbai,Full-time,Tech,Software Developer,Cere Labs,"₹300,000 - ₹400,000",about us cere labs is a mumbai based company w...,"Mumbai, Maharashtra",2026-01-24T11:20:47Z,Entry Level (0-2 years),...,Real Job,software developer cere labs tech entry level ...,0,541,93,4.827957,0,1,www.adzuna.in,6553
2,Mumbai,Full-time,Tech,Software Developer,Image Media World,"₹500,000 - ₹700,000",job description software developer position ov...,"Mumbai, Maharashtra",2024-04-03T18:05:16Z,Entry Level (0-2 years),...,Real Job,software developer image media world tech entr...,0,549,78,6.051282,0,0,www.adzuna.in,6553
3,Mumbai,Full-time,Tech,Software Developer,Ion,Not specified,job summary writes new software makes modifica...,"Mumbai, Maharashtra",2026-05-17T06:47:16Z,Entry Level (0-2 years),...,Real Job,software developer ion tech entry level years ...,0,526,71,6.422535,0,0,www.adzuna.in,6553
4,Mumbai,Full-time,Tech,Software Developer,Etaash Consultants,"₹300,000 - ₹1,500,000",job opening in it company for angular develope...,"Mumbai, Maharashtra",2021-09-03T01:56:01Z,Entry Level (0-2 years),...,Real Job,software developer etaash consultants tech ent...,0,537,79,5.810127,0,0,www.adzuna.in,6553


In [15]:
# Remove Duplicate Job Posts

print("Before:", df.shape)

df = df.drop_duplicates(subset=["combined_text"]).reset_index(drop=True)

print("After :", df.shape)

print("Remaining Duplicates:",
      df["combined_text"].duplicated().sum())

Before: (7419, 21)
After : (7345, 21)
Remaining Duplicates: 0


Section 3 : Train-Test Split

In [17]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (5876, 21)
Test Shape : (1469, 21)


Section 4 : Fit TF-IDF on Train data only

In [18]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# Fit only on training data
X_train_tfidf = tfidf.fit_transform(
    train_df["combined_text"]
)

# Transform test data
X_test_tfidf = tfidf.transform(
    test_df["combined_text"]
)

print("X_train_tfidf Shape:", X_train_tfidf.shape)
print("X_test_tfidf Shape :", X_test_tfidf.shape)

X_train_tfidf Shape: (5876, 5000)
X_test_tfidf Shape : (1469, 5000)


Section 5 : Saving TF-IDF vectorizer

In [19]:
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer Saved")

TF-IDF Vectorizer Saved


Section 6 : Extract numerical Features

In [20]:
numerical_features = [
    "text_length",
    "word_count",
    "avg_word_length",
    "keyword_risk_score",
    "salary_risk_score",
    "domain_frequency"
]

# Train numerical features
X_train_num = csr_matrix(
    train_df[numerical_features].values
)

# Test numerical features
X_test_num = csr_matrix(
    test_df[numerical_features].values
)

print("X_train_num Shape:", X_train_num.shape)
print("X_test_num Shape :", X_test_num.shape)

X_train_num Shape: (5876, 6)
X_test_num Shape : (1469, 6)


Section 7 : Final Feature Matrix after combining TF-IDF + Numerical Features

In [21]:
X_train = hstack([X_train_tfidf, X_train_num])

X_test = hstack([X_test_tfidf, X_test_num])

print("Final X_train Shape:", X_train.shape)
print("Final X_test Shape :", X_test.shape)

Final X_train Shape: (5876, 5006)
Final X_test Shape : (1469, 5006)


Section 8 : Creating Target Variables

In [22]:
y_train = train_df["label"]

y_test = test_df["label"]

print("y_train Shape:", y_train.shape)
print("y_test Shape :", y_test.shape)

print("\nTrain Distribution:")
print(y_train.value_counts())

print("\nTest Distribution:")
print(y_test.value_counts())

y_train Shape: (5876,)
y_test Shape : (1469,)

Train Distribution:
label
0    5221
1     655
Name: count, dtype: int64

Test Distribution:
label
0    1305
1     164
Name: count, dtype: int64


Section 9 : Saving Prepared Data


In [23]:
from pathlib import Path
from scipy.sparse import save_npz

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save sparse matrices
save_npz(output_dir / "X_train.npz", X_train)
save_npz(output_dir / "X_test.npz", X_test)

# Save targets
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("Model Preparation Completed")

Model Preparation Completed


In [24]:
# 1. Duplicate texts in entire dataset
print(df["combined_text"].duplicated().sum())

0


In [25]:
# 2. Common texts between train and test
train_set = set(train_df["combined_text"])
test_set = set(test_df["combined_text"])

print(len(train_set.intersection(test_set)))

0
